# Kaggle 03 - Evaluation / Ablation

Run benchmark and ablation after the Qdrant Cloud index is populated.

Kaggle settings:
- Internet: On
- Accelerator: GPU optional for eval; CPU is acceptable for lightweight runs
- Secrets: `OPENROUTER_API_KEY`, `QDRANT_URL`, `QDRANT_API_KEY`


In [ ]:
REPO_URL = "https://github.com/phamdinhhai/project-ks2.git"
PROJECT_DIR = "/kaggle/working/project-ks2"

import os
if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}
!git pull --ff-only


In [ ]:
!python -m pip install -U pip
!pip install -e ".[qdrant,agent,eval]"
!pip install requests


In [ ]:
# Load Kaggle Secrets. Add these in Notebook > Add-ons > Secrets.
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for name in ["OPENROUTER_API_KEY", "QDRANT_URL", "QDRANT_API_KEY"]:
    try:
        value = secrets.get_secret(name)
    except Exception as exc:
        value = None
        print(f"Secret {name} unavailable: {exc}")
    if value:
        os.environ[name] = value

os.environ.setdefault("OPENROUTER_MODEL", "google/gemini-2.5-flash")
os.environ.setdefault("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
os.environ.setdefault("QDRANT_TEXT_COLLECTION", "text_chunks_prod")
os.environ.setdefault("QDRANT_IMAGE_COLLECTION", "image_patches_prod")
os.environ.setdefault("QDRANT_INDEX_STATE", "/kaggle/working/outputs/index_state/kaggle_index_state.json")

print("OPENROUTER_API_KEY set:", bool(os.environ.get("OPENROUTER_API_KEY")))
print("OPENROUTER_MODEL:", os.environ.get("OPENROUTER_MODEL"))
print("QDRANT_URL set:", bool(os.environ.get("QDRANT_URL")))
print("QDRANT_API_KEY set:", bool(os.environ.get("QDRANT_API_KEY")))
print("Text collection:", os.environ.get("QDRANT_TEXT_COLLECTION"))
print("Image collection:", os.environ.get("QDRANT_IMAGE_COLLECTION"))


In [ ]:
# Provider diagnostics
!python -m medical_rag test-openrouter
!python -m medical_rag test-qdrant --qdrant-url "$QDRANT_URL" --use-cloud-auth


In [ ]:
# Baseline advanced metrics
!python scripts/colab_workflow.py eval-baseline   --eval-file data/eval_cases.json   --output-file outputs/benchmark/baseline_advanced.json


In [ ]:
# Agent metrics with Qdrant Cloud + OpenRouter
!python scripts/colab_workflow.py eval-agent   --eval-file data/eval_cases.json   --output-file outputs/benchmark/agent_openrouter.json   --use-qdrant


In [ ]:
# Ablation report
!python scripts/colab_workflow.py ablation   --eval-file data/eval_cases.json   --output-dir outputs/ablation


In [ ]:
# Copy outputs to Kaggle working artifacts for download.
!mkdir -p /kaggle/working/artifacts
!cp -r outputs /kaggle/working/artifacts/outputs
!find /kaggle/working/artifacts -maxdepth 4 -type f | sort
